# MLDS 2026 Assignment 2 - Training v6 (Focused & Reliable)
## Key v6 changes vs v5:
- **Classification**: Back to BCEWithLogitsLoss (ASL was unstable), 256px (not 320), moderate augmentation, NO Mixup/CutMix, NO SWA, batch=32, faster convergence
- **Segmentation**: Same ResNet101+ASPP+UNet, moderate augmentation, Dice+CE loss, keep EMA, no SWA
- **Philosophy**: Fewer moving parts = better debugging, faster convergence, higher scores

In [1]:
# ---- Cell 1: Setup & Dataset ----
!pip install -q albumentations

import os

DATA_ROOT = '/kaggle/input/datasets/adityapatel2004/mldataseta2/Dataset'

print(f'DATA_ROOT = {DATA_ROOT}')
print('Train images:', len(os.listdir(os.path.join(DATA_ROOT, 'train', 'images'))))
print('Test images:', len(os.listdir(os.path.join(DATA_ROOT, 'test', 'images'))))

DATA_ROOT = /kaggle/input/datasets/adityapatel2004/mldataseta2/Dataset
Train images: 2200
Test images: 713


In [2]:
# ---- Cell 2: Imports ----
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torch.amp as amp
import matplotlib.pyplot as plt
import copy, time, random, os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [3]:
# ---- Cell 3: Paths & Split ----
TRAIN_IMG = os.path.join(DATA_ROOT, 'train', 'images')
TRAIN_MASK = os.path.join(DATA_ROOT, 'train', 'segmentation_masks')
LABELS_CSV = os.path.join(DATA_ROOT, 'train', 'labels.csv')
TEST_IMG = os.path.join(DATA_ROOT, 'test', 'images')

CLASS_NAMES = ['aeroplane','bicycle','bird','boat','bottle','bus','car','cat',
               'chair','cow','diningtable','dog','horse','motorbike','person',
               'pottedplant','sheep','sofa','train','tvmonitor']

labels_df = pd.read_csv(LABELS_CSV)
all_ids = labels_df['image_id'].tolist()

train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
print(f'Train: {len(train_ids)}, Val: {len(val_ids)}')

Train: 1760, Val: 440


---
## Part 1: Multi-Label Classification

In [4]:
# ---- Cell 5: Classification Dataset (v6: 256px + moderate aug) ----
CLS_SIZE = 256

class ClsDataset(Dataset):
    def __init__(self, img_ids, labels_df, img_dir, transform=None):
        self.img_ids = img_ids
        self.img_dir = img_dir
        self.transform = transform
        self.labels_df = labels_df.set_index('image_id')

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, f'{img_id}.jpg')).convert('RGB'))
        label = self.labels_df.loc[img_id, CLASS_NAMES].values.astype(np.float32)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label)

cls_train_tf = A.Compose([
    A.RandomResizedCrop(size=(CLS_SIZE, CLS_SIZE), scale=(0.65, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=15, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=3, p=1.0),
        A.MedianBlur(blur_limit=3, p=1.0),
    ], p=0.2),
    A.CoarseDropout(max_holes=6, max_height=CLS_SIZE//10, max_width=CLS_SIZE//10, p=0.25),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

cls_val_tf = A.Compose([
    A.Resize(height=CLS_SIZE + 32, width=CLS_SIZE + 32),
    A.CenterCrop(height=CLS_SIZE, width=CLS_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

cls_train_ds = ClsDataset(train_ids, labels_df, TRAIN_IMG, cls_train_tf)
cls_val_ds = ClsDataset(val_ids, labels_df, TRAIN_IMG, cls_val_tf)
cls_train_dl = DataLoader(cls_train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
cls_val_dl = DataLoader(cls_val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Classification batches: train={len(cls_train_dl)}, val={len(cls_val_dl)}')


Classification batches: train=55, val=14


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/3542552886.py:31: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=CLS_SIZE//10, max_width=CLS_SIZE//10, p=0.25),


In [5]:
# ---- Cell 6: Classification Model with GeM Pooling ----
class GeM(nn.Module):
    """Generalized Mean Pooling - better than avg pooling for retrieval/classification"""
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)

class MultiLabelClassifier(nn.Module):
    def __init__(self, num_classes=20, pretrained=True):
        super().__init__()
        if pretrained:
            backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        else:
            backbone = models.resnet50(weights=None)
        # Use all layers except avgpool and fc
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.gem = GeM(p=3.0)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )
        # Freeze backbone initially
        for p in self.features.parameters():
            p.requires_grad = False

    def unfreeze_top(self):
        """Unfreeze layer3 and layer4 only"""
        for p in self.features[5:].parameters():  # layer2 onwards
            p.requires_grad = True

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

    def forward(self, x):
        feat = self.features(x)
        pooled = self.gem(feat)
        return self.head(pooled)

cls_model = MultiLabelClassifier(num_classes=20, pretrained=True).to(device)
trainable = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in cls_model.parameters())
print(f'Cls model: {trainable/1e6:.1f}M trainable / {total/1e6:.1f}M total')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 247MB/s]


Cls model: 1.2M trainable / 24.7M total


In [6]:
# ---- Cell 7: Loss (v6: BCEWithLogitsLoss + pos_weight) ----
class_counts = labels_df[CLASS_NAMES].sum().values
pos_weight = torch.tensor(
    (len(labels_df) - class_counts) / np.clip(class_counts, 1, None),
    dtype=torch.float32
).to(device)
pos_weight = pos_weight.clamp(max=15.0)
print('pos_weight (clamped at 15):', pos_weight.cpu().numpy().round(1))


pos_weight (clamped at 15): [15.  15.  12.7 15.  14.4 15.  10.3 10.6  9.9 15.  15.  10.8 15.  15.
  2.3 15.  15.  14.7 15.  15. ]


In [7]:
# ---- Cell 8: Train Classification v6 ----
class EMA:
    def __init__(self, model, decay=0.998):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v
    def apply(self, model):
        model.load_state_dict(self.shadow)

def train_classifier_v6(model, train_dl, val_dl, pos_weight, epochs=60,
                         unfreeze_top_epoch=3, full_unfreeze_epoch=8):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = amp.GradScaler()

    head_params = list(model.head.parameters()) + list(model.gem.parameters())
    optimizer = torch.optim.AdamW(head_params, lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=2e-3, steps_per_epoch=len(train_dl), epochs=unfreeze_top_epoch
    )

    ema = EMA(model, decay=0.998)
    best_f1, best_state = 0, None
    history = {'train_loss': [], 'val_f1': []}
    patience, patience_counter = 15, 0

    for epoch in range(epochs):
        if epoch == unfreeze_top_epoch:
            model.unfreeze_top()
            optimizer = torch.optim.AdamW([
                {'params': model.features[5:].parameters(), 'lr': 5e-5},
                {'params': head_params, 'lr': 1e-3}
            ], weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=[5e-5, 1e-3],
                steps_per_epoch=len(train_dl), epochs=full_unfreeze_epoch - unfreeze_top_epoch
            )
            print(f'  >> Phase 2: Unfreezing top layers at epoch {epoch}')

        if epoch == full_unfreeze_epoch:
            model.unfreeze_all()
            optimizer = torch.optim.AdamW([
                {'params': model.features[:5].parameters(), 'lr': 5e-6},
                {'params': model.features[5:].parameters(), 'lr': 3e-5},
                {'params': head_params, 'lr': 3e-4}
            ], weight_decay=1e-4)
            remaining = epochs - full_unfreeze_epoch
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=remaining, eta_min=1e-6
            )
            print(f'  >> Phase 3: Unfreezing all layers at epoch {epoch} (cosine for {remaining} epochs)')

        model.train()
        running_loss = 0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            with amp.autocast(device_type='cuda'):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)
            if epoch < full_unfreeze_epoch:
                scheduler.step()
            running_loss += loss.item() * imgs.size(0)

        if epoch >= full_unfreeze_epoch:
            scheduler.step()

        epoch_loss = running_loss / len(train_dl.dataset)

        orig_state = copy.deepcopy(model.state_dict())
        ema.apply(model)
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs = imgs.to(device)
                with amp.autocast(device_type='cuda'):
                    probs = torch.sigmoid(model(imgs)).float().cpu().numpy()
                all_preds.append(probs)
                all_labels.append(labels.numpy())
        all_preds_np = np.vstack(all_preds)
        all_labels_np = np.vstack(all_labels)
        val_f1 = f1_score(all_labels_np, (all_preds_np >= 0.5).astype(int), average='samples', zero_division=0)
        model.load_state_dict(orig_state)

        history['train_loss'].append(epoch_loss)
        history['val_f1'].append(val_f1)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy({k: v.clone() for k, v in ema.shadow.items()})
            patience_counter = 0
            os.makedirs('/kaggle/working/output/classification/weights', exist_ok=True)
            torch.save(best_state, '/kaggle/working/output/classification/weights/best_model.pth')
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d}/{epochs} | loss={epoch_loss:.4f} | val_f1={val_f1:.4f} | best={best_f1:.4f}')

        if patience_counter >= patience and epoch >= full_unfreeze_epoch + 10:
            print(f'  >> Early stopping at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs = imgs.to(device)
            with amp.autocast(device_type='cuda'):
                probs = torch.sigmoid(model(imgs)).float().cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.numpy())

    return model, history, np.vstack(all_preds), np.vstack(all_labels)

print('Training classifier v6...')
t0 = time.time()
cls_model, cls_hist, val_preds, val_labels = train_classifier_v6(
    cls_model, cls_train_dl, cls_val_dl, pos_weight, epochs=60,
    unfreeze_top_epoch=3, full_unfreeze_epoch=8
)
print(f'Done in {(time.time()-t0)/60:.1f} min | Best F1: {max(cls_hist["val_f1"]):.4f}')


Training classifier v6...
Epoch   1/60 | loss=0.8454 | val_f1=0.1965 | best=0.1965
  >> Phase 2: Unfreezing top layers at epoch 3
Epoch   5/60 | loss=0.2575 | val_f1=0.6544 | best=0.6544
  >> Phase 3: Unfreezing all layers at epoch 8 (cosine for 52 epochs)
Epoch  10/60 | loss=0.1401 | val_f1=0.7624 | best=0.7624
Epoch  15/60 | loss=0.0927 | val_f1=0.8187 | best=0.8187
Epoch  20/60 | loss=0.0639 | val_f1=0.8483 | best=0.8483
Epoch  25/60 | loss=0.0479 | val_f1=0.8657 | best=0.8657
Epoch  30/60 | loss=0.0424 | val_f1=0.8694 | best=0.8695
Epoch  35/60 | loss=0.0325 | val_f1=0.8740 | best=0.8755
Epoch  40/60 | loss=0.0257 | val_f1=0.8711 | best=0.8755
Epoch  45/60 | loss=0.0239 | val_f1=0.8714 | best=0.8755
  >> Early stopping at epoch 49
Done in 9.9 min | Best F1: 0.8755


In [8]:
# ---- Cell 9: Optimal per-class thresholds (finer search) ----
# Cast to float32 — AMP produces float16; scipy sparse does not support float16
val_preds  = val_preds.astype(np.float32)
val_labels = val_labels.astype(np.float32)

best_thresholds = []
for c in range(20):
    best_f1_c, best_t = 0, 0.5
    for t in np.arange(0.10, 0.91, 0.02):   # finer grid
        preds_c = (val_preds[:, c] >= t).astype(np.int32)
        f1_c = f1_score(val_labels[:, c], preds_c, zero_division=0)
        if f1_c > best_f1_c:
            best_f1_c, best_t = f1_c, t
    best_thresholds.append(best_t)
    print(f'{CLASS_NAMES[c]:15s}: threshold={best_t:.2f}, f1={best_f1_c:.3f}')

# Use int32 for opt_preds — avoids scipy float16/float32 sparse issues
opt_preds = np.zeros_like(val_preds, dtype=np.int32)
for c in range(20):
    opt_preds[:, c] = (val_preds[:, c] >= best_thresholds[c]).astype(np.int32)

opt_f1 = f1_score(val_labels.astype(np.int32), opt_preds, average='samples', zero_division=0)
print(f'\nOptimized F1 (per-class thresholds): {opt_f1:.4f}')

aeroplane      : threshold=0.18, f1=1.000
bicycle        : threshold=0.86, f1=0.824
bird           : threshold=0.46, f1=0.972
boat           : threshold=0.68, f1=0.957
bottle         : threshold=0.84, f1=0.873
bus            : threshold=0.88, f1=0.933
car            : threshold=0.58, f1=0.854
cat            : threshold=0.72, f1=0.952
chair          : threshold=0.66, f1=0.682
cow            : threshold=0.82, f1=0.927
diningtable    : threshold=0.90, f1=0.750
dog            : threshold=0.78, f1=0.958
horse          : threshold=0.80, f1=0.909
motorbike      : threshold=0.50, f1=0.927
person         : threshold=0.48, f1=0.881
pottedplant    : threshold=0.42, f1=0.634
sheep          : threshold=0.50, f1=0.950
sofa           : threshold=0.48, f1=0.719
train          : threshold=0.88, f1=0.978
tvmonitor      : threshold=0.90, f1=0.875

Optimized F1 (per-class thresholds): 0.8973


In [9]:
# ---- Cell 10: Save classifier weights ----
os.makedirs('output/classification/weights', exist_ok=True)
torch.save(cls_model.state_dict(), 'output/classification/weights/best_model.pth')
np.save('output/classification/weights/thresholds.npy', np.array(best_thresholds))
print('Classification weights saved')

Classification weights saved


---
## Part 2: Semantic Segmentation

In [10]:
# ---- Cell 12: Segmentation Dataset (v6: moderate augmentation) ----
class SegDataset(Dataset):
    def __init__(self, img_ids, img_dir, mask_dir, transform=None):
        self.img_ids = [i for i in img_ids if os.path.exists(os.path.join(mask_dir, f'{i}.png'))]
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, f'{img_id}.jpg')).convert('RGB'))
        mask = np.array(Image.open(os.path.join(self.mask_dir, f'{img_id}.png')))
        if self.transform:
            out = self.transform(image=img, mask=mask)
            img, mask = out['image'], out['mask']
        return img, mask.long()

SEG_SIZE = 512

seg_train_tf = A.Compose([
    A.RandomResizedCrop(size=(SEG_SIZE, SEG_SIZE), scale=(0.5, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=15, border_mode=0, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.4),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.3),
    A.GaussianBlur(blur_limit=3, p=0.15),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

seg_val_tf = A.Compose([
    A.Resize(height=SEG_SIZE, width=SEG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

seg_train_ds = SegDataset(train_ids, TRAIN_IMG, TRAIN_MASK, seg_train_tf)
seg_val_ds = SegDataset(val_ids, TRAIN_IMG, TRAIN_MASK, seg_val_tf)
seg_train_dl = DataLoader(seg_train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
seg_val_dl = DataLoader(seg_val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f'Segmentation: train={len(seg_train_ds)}, val={len(seg_val_ds)}')


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Segmentation: train=1760, val=440


In [11]:
# ---- Cell 13: Segmentation Model v5 (ResNet101 + ASPP + UNet + Aux) ----
class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256):
        super().__init__()
        self.conv1x1 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.conv_r6 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=6, dilation=6), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.conv_r12 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=12, dilation=12), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.conv_r18 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=18, dilation=18), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.project = nn.Sequential(nn.Conv2d(out_ch * 5, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout(0.5))
    def forward(self, x):
        h, w = x.shape[2:]
        p = F.interpolate(self.pool(x), size=(h, w), mode='bilinear', align_corners=False)
        return self.project(torch.cat([self.conv1x1(x), self.conv_r6(x), self.conv_r12(x), self.conv_r18(x), p], dim=1))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class SegResNet101UNet(nn.Module):
    """v5: ResNet101 backbone for more capacity + ASPP + UNet decoder"""
    def __init__(self, num_classes=21, pretrained=True):
        super().__init__()
        if pretrained:
            backbone = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        else:
            backbone = models.resnet101(weights=None)

        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.aspp = ASPP(2048, 256)
        self.dec4 = DecoderBlock(256, 1024, 256)
        self.dec3 = DecoderBlock(256, 512, 128)
        self.dec2 = DecoderBlock(128, 256, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.final_up = nn.ConvTranspose2d(32, 32, kernel_size=4, stride=2, padding=1)
        self.final_conv = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        # Auxiliary head from layer3
        self.aux_head = nn.Sequential(
            nn.Conv2d(1024, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Conv2d(256, num_classes, 1)
        )

        # Freeze ALL encoder initially
        for p in self.layer0.parameters(): p.requires_grad = False
        for p in self.pool.parameters(): p.requires_grad = False
        for p in self.layer1.parameters(): p.requires_grad = False
        for p in self.layer2.parameters(): p.requires_grad = False
        for p in self.layer3.parameters(): p.requires_grad = False
        for p in self.layer4.parameters(): p.requires_grad = False

    def unfreeze_top(self):
        for p in self.layer3.parameters(): p.requires_grad = True
        for p in self.layer4.parameters(): p.requires_grad = True

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True

    def forward(self, x):
        e0 = self.layer0(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        x = self.aspp(e4)
        x = self.dec4(x, e3)
        x = self.dec3(x, e2)
        x = self.dec2(x, e1)
        x = self.dec1(x, e0)
        x = F.relu(self.final_up(x))
        main_out = self.final_conv(x)
        if self.training:
            aux_out = self.aux_head(e3)
            return main_out, aux_out
        return main_out

seg_model = SegResNet101UNet(num_classes=21, pretrained=True).to(device)
trainable = sum(p.numel() for p in seg_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in seg_model.parameters())
print(f'Seg model (ResNet101): {trainable/1e6:.1f}M trainable / {total/1e6:.1f}M total')

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 129MB/s]  


Seg model (ResNet101): 24.4M trainable / 66.9M total


In [12]:
# ---- Cell 14: Loss v6 (Dice + CE) & mIoU ----
class DiceLoss(nn.Module):
    def __init__(self, num_classes=21, ignore_index=255, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, inputs, targets):
        valid = (targets != self.ignore_index)
        targets_clean = targets.clone()
        targets_clean[~valid] = 0
        probs = F.softmax(inputs, dim=1)
        one_hot = F.one_hot(targets_clean, self.num_classes).permute(0, 3, 1, 2).float()
        valid_mask = valid.unsqueeze(1).float()
        probs = probs * valid_mask
        one_hot = one_hot * valid_mask
        dims = (0, 2, 3)
        intersection = (probs * one_hot).sum(dim=dims)
        cardinality = (probs + one_hot).sum(dim=dims)
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return (1 - dice).mean()

class DiceCELoss(nn.Module):
    def __init__(self, num_classes=21, ignore_index=255, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice = DiceLoss(num_classes, ignore_index)
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight

    def forward(self, inputs, targets):
        return self.dice_weight * self.dice(inputs, targets) + self.ce_weight * self.ce(inputs, targets)

def compute_miou_dataset(all_preds, all_targets, num_classes=21, ignore=255):
    intersection = np.zeros(num_classes)
    union = np.zeros(num_classes)
    for preds, targets in zip(all_preds, all_targets):
        valid = targets != ignore
        for c in range(num_classes):
            p = (preds == c) & valid
            g = (targets == c) & valid
            intersection[c] += (p & g).sum()
            union[c] += (p | g).sum()
    ious = [intersection[c] / union[c] for c in range(num_classes) if union[c] > 0]
    return np.mean(ious) if ious else 0.0

print('DiceCE Loss and mIoU functions defined')


DiceCE Loss and mIoU functions defined


In [13]:
# ---- Cell 15: Train Segmentation v6 (Dice+CE, PolyLR, NO EMA) ----
def poly_lr_lambda(epoch, max_epochs, power=0.9):
    return max((1 - epoch / max_epochs) ** power, 1e-6)

def train_segmentation_v6(model, train_dl, val_dl, epochs=70,
                           unfreeze_top_epoch=3, unfreeze_all_epoch=8,
                           accum_steps=2):
    criterion = DiceCELoss(num_classes=21, ignore_index=255)
    aux_criterion = nn.CrossEntropyLoss(ignore_index=255)
    scaler = amp.GradScaler()

    decoder_params = (list(model.aspp.parameters()) + list(model.dec4.parameters()) +
                      list(model.dec3.parameters()) + list(model.dec2.parameters()) +
                      list(model.dec1.parameters()) + list(model.final_up.parameters()) +
                      list(model.final_conv.parameters()) + list(model.aux_head.parameters()))
    optimizer = torch.optim.AdamW(decoder_params, lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=2e-3,
        steps_per_epoch=len(train_dl) // accum_steps + 1,
        epochs=unfreeze_top_epoch
    )

    best_miou, best_state = 0, None
    history = {'train_loss': [], 'val_miou': []}
    patience, patience_counter = 18, 0

    for epoch in range(epochs):
        if epoch == unfreeze_top_epoch:
            model.unfreeze_top()
            optimizer = torch.optim.AdamW([
                {'params': list(model.layer3.parameters()) + list(model.layer4.parameters()), 'lr': 5e-5},
                {'params': decoder_params, 'lr': 1e-3}
            ], weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=[5e-5, 1e-3],
                steps_per_epoch=len(train_dl) // accum_steps + 1,
                epochs=unfreeze_all_epoch - unfreeze_top_epoch
            )
            print(f'  >> Phase 2: Unfreezing layer3+layer4 at epoch {epoch}')

        if epoch == unfreeze_all_epoch:
            model.unfreeze_all()
            low_enc = (list(model.layer0.parameters()) + list(model.layer1.parameters()) +
                       list(model.layer2.parameters()))
            top_enc = list(model.layer3.parameters()) + list(model.layer4.parameters())
            optimizer = torch.optim.AdamW([
                {'params': low_enc,        'lr': 5e-6},
                {'params': top_enc,        'lr': 3e-5},
                {'params': decoder_params, 'lr': 5e-4}
            ], weight_decay=1e-4)
            remaining = epochs - unfreeze_all_epoch
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer,
                lr_lambda=lambda ep: poly_lr_lambda(ep, remaining, power=0.9)
            )
            print(f'  >> Phase 3: Unfreezing ALL at epoch {epoch} (PolyLR, {remaining} epochs left)')

        aux_weight = max(0.1, 0.4 * (1.0 - epoch / epochs))

        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        for batch_idx, (imgs, masks) in enumerate(train_dl):
            imgs, masks = imgs.to(device), masks.to(device)
            with amp.autocast(device_type='cuda'):
                main_out, aux_out = model(imgs)
                main_loss = criterion(main_out, masks)
                aux_up = F.interpolate(aux_out, size=masks.shape[1:], mode='bilinear', align_corners=False)
                aux_loss = aux_criterion(aux_up, masks)
                loss = (main_loss + aux_weight * aux_loss) / accum_steps
            scaler.scale(loss).backward()
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if epoch < unfreeze_all_epoch:
                    scheduler.step()
            running_loss += loss.item() * accum_steps * imgs.size(0)

        if epoch >= unfreeze_all_epoch:
            scheduler.step()

        epoch_loss = running_loss / len(train_dl.dataset)

        # Validate with actual model (NO EMA — decoder starts random, EMA stays random too long)
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs = imgs.to(device)
                with amp.autocast(device_type='cuda'):
                    preds = model(imgs).argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(masks.numpy())
        val_miou = compute_miou_dataset(all_preds, all_targets)

        history['train_loss'].append(epoch_loss)
        history['val_miou'].append(val_miou)

        if val_miou > best_miou:
            best_miou = val_miou
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            os.makedirs('/kaggle/working/output/segmentation/weights', exist_ok=True)
            torch.save(best_state, '/kaggle/working/output/segmentation/weights/best_model.pth')

        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d}/{epochs} | loss={epoch_loss:.4f} | val_mIoU={val_miou:.4f} | best={best_miou:.4f}')

        if patience_counter >= patience and epoch >= unfreeze_all_epoch + 12:
            print(f'  >> Early stopping at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    return model, history

print('Training segmentation v6...')
t0 = time.time()
seg_model, seg_hist = train_segmentation_v6(
    seg_model, seg_train_dl, seg_val_dl, epochs=70,
    unfreeze_top_epoch=3, unfreeze_all_epoch=8, accum_steps=2
)
print(f'Done in {(time.time()-t0)/60:.1f} min | Best mIoU: {max(seg_hist["val_miou"]):.4f}')


Training segmentation v6...
Epoch   1/70 | loss=2.0618 | val_mIoU=0.0654 | best=0.0654
  >> Phase 2: Unfreezing layer3+layer4 at epoch 3
Epoch   5/70 | loss=1.1183 | val_mIoU=0.0854 | best=0.0854
  >> Phase 3: Unfreezing ALL at epoch 8 (PolyLR, 62 epochs left)
Epoch  10/70 | loss=0.9392 | val_mIoU=0.1242 | best=0.1242
Epoch  15/70 | loss=0.7800 | val_mIoU=0.1285 | best=0.1866
Epoch  20/70 | loss=0.6748 | val_mIoU=0.3802 | best=0.4045
Epoch  25/70 | loss=0.5720 | val_mIoU=0.6232 | best=0.6232
Epoch  30/70 | loss=0.4988 | val_mIoU=0.6796 | best=0.6796
Epoch  35/70 | loss=0.4589 | val_mIoU=0.6298 | best=0.6824
Epoch  40/70 | loss=0.4241 | val_mIoU=0.7255 | best=0.7255
Epoch  45/70 | loss=0.3939 | val_mIoU=0.7180 | best=0.7366
Epoch  50/70 | loss=0.3836 | val_mIoU=0.7471 | best=0.7471
Epoch  55/70 | loss=0.3667 | val_mIoU=0.7344 | best=0.7519
Epoch  60/70 | loss=0.3579 | val_mIoU=0.7522 | best=0.7601
Epoch  65/70 | loss=0.3526 | val_mIoU=0.7606 | best=0.7635
Epoch  70/70 | loss=0.3514 | va

In [14]:
# ---- Cell 16: Save segmentation weights ----
os.makedirs('output/segmentation/weights', exist_ok=True)
torch.save(seg_model.state_dict(), 'output/segmentation/weights/best_model.pth')
print('Segmentation weights saved')

Segmentation weights saved


In [ ]:
# ---- Cell: Generate Training Curves ----
import matplotlib.pyplot as plt
import numpy as np

# Classification training data (from training output above)
cls_epochs = [1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 49]
cls_loss =   [0.8454, 0.2575, 0.1401, 0.0927, 0.0639, 0.0479, 0.0424, 0.0325, 0.0257, 0.0239, 0.0220]
cls_f1 =     [0.1965, 0.6544, 0.7624, 0.8187, 0.8483, 0.8657, 0.8694, 0.8740, 0.8711, 0.8714, 0.8755]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(cls_epochs, cls_loss, 'o-', color='#d62728', linewidth=1.5, markersize=4)
ax1.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='Unfreeze top (ep 3)')
ax1.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='Unfreeze all (ep 8)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss'); ax1.set_title('Classification Loss')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(cls_epochs, cls_f1, 's-', color='#2ca02c', linewidth=1.5, markersize=4)
ax2.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='Unfreeze top (ep 3)')
ax2.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='Unfreeze all (ep 8)')
ax2.axhline(y=0.8755, color='#2ca02c', linestyle='--', alpha=0.4, label=f'Best F1={0.8755:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation F1'); ax2.set_title('Classification F1 Score')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1.0)
plt.tight_layout(); plt.savefig('fig1_classification_curves.png', dpi=150, bbox_inches='tight'); plt.show()

# Segmentation training data
seg_epochs = [1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
seg_loss =   [2.0618, 1.1183, 0.9392, 0.7800, 0.6748, 0.5720, 0.4988, 0.4589, 0.4241, 0.3939, 0.3836, 0.3667, 0.3579, 0.3526, 0.3514]
seg_miou =   [0.0654, 0.0854, 0.1242, 0.1285, 0.3802, 0.6232, 0.6796, 0.6298, 0.7255, 0.7180, 0.7471, 0.7344, 0.7522, 0.7606, 0.7638]
seg_best =   [0.0654, 0.0854, 0.1242, 0.1866, 0.4045, 0.6232, 0.6796, 0.6824, 0.7255, 0.7366, 0.7471, 0.7519, 0.7601, 0.7635, 0.7684]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(seg_epochs, seg_loss, 'o-', color='#d62728', linewidth=1.5, markersize=4)
ax1.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='Unfreeze L3+L4 (ep 3)')
ax1.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='Unfreeze all (ep 8)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss (DiceCE)'); ax1.set_title('Segmentation Loss')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(seg_epochs, seg_miou, 's-', color='#1f77b4', linewidth=1.5, markersize=4, label='Val mIoU')
ax2.plot(seg_epochs, seg_best, '--', color='#1f77b4', alpha=0.5, linewidth=1, label='Best mIoU')
ax2.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='Unfreeze L3+L4 (ep 3)')
ax2.axvline(x=8, color='gray', linestyle=':', alpha=0.5, label='Unfreeze all (ep 8)')
ax2.axhline(y=0.7684, color='#1f77b4', linestyle='--', alpha=0.4, label=f'Best={0.7684:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation mIoU'); ax2.set_title('Segmentation mIoU')
ax2.legend(fontsize=8, loc='lower right'); ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 0.85)
plt.tight_layout(); plt.savefig('fig2_segmentation_curves.png', dpi=150, bbox_inches='tight'); plt.show()
print('Plots saved!')


---
## Part 3: Generate submission.csv with Enhanced TTA

In [15]:
# ---- Cell 18: RLE encoding ----
def mask_to_rle(mask):
    flat = mask.flatten()
    starts, lengths, values = [], [], []
    i = 0
    while i < len(flat):
        if flat[i] != 0:
            val = flat[i]
            start = i
            while i < len(flat) and flat[i] == val:
                i += 1
            starts.append(start)
            lengths.append(i - start)
            values.append(val)
        else:
            i += 1
    if not starts:
        return ''
    parts = []
    for s, l, v in zip(starts, lengths, values):
        parts.extend([str(s), str(l), str(v)])
    return ' '.join(parts)

print('RLE encoder defined')

RLE encoder defined


In [16]:
# ---- Cell 19: TTA functions (v6) ----
def cls_predict_tta(model, image_np, thresholds):
    """TTA: hflip × 5 scales"""
    model.eval()
    all_probs = []
    for flip in [False, True]:
        img = np.fliplr(image_np).copy() if flip else image_np
        for size in [CLS_SIZE - 32, CLS_SIZE, CLS_SIZE + 32, CLS_SIZE + 64]:
            resize_to = max(size, CLS_SIZE)
            tf = A.Compose([
                A.Resize(height=resize_to, width=resize_to),
                A.CenterCrop(height=CLS_SIZE, width=CLS_SIZE),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
            inp = tf(image=img)['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                with amp.autocast(device_type='cuda'):
                    probs = torch.sigmoid(model(inp)).float().squeeze().cpu().numpy()
            all_probs.append(probs)
    avg = np.mean(all_probs, axis=0)
    classes = [CLASS_NAMES[i] for i in range(20) if avg[i] >= thresholds[i]]
    return ' '.join(classes) if classes else 'background'

def seg_predict_tta(model, image_np, orig_h, orig_w):
    """TTA: hflip × multi-scale, average logits"""
    model.eval()
    logits_sum = None
    count = 0
    for flip in [False, True]:
        img = np.fliplr(image_np).copy() if flip else image_np
        for scale in [0.75, 1.0, 1.25, 1.5]:
            sz = int(SEG_SIZE * scale)
            tf = A.Compose([
                A.Resize(height=sz, width=sz),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
            inp = tf(image=img)['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                with amp.autocast(device_type='cuda'):
                    out = model(inp)
                out = F.interpolate(out, size=(SEG_SIZE, SEG_SIZE), mode='bilinear', align_corners=False)
            if flip:
                out = torch.flip(out, dims=[3])
            logits_sum = out if logits_sum is None else logits_sum + out
            count += 1
    avg_logits = logits_sum / count
    pred = avg_logits.argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    pred_resized = np.array(Image.fromarray(pred).resize((orig_w, orig_h), Image.NEAREST))
    return pred_resized

print('TTA functions defined (v6)')

TTA functions defined (v6)


In [17]:
# ---- Cell 20: Generate submission ----
test_ids = sorted([f.replace('.jpg', '') for f in os.listdir(TEST_IMG) if f.endswith('.jpg')])
print(f'Test images: {len(test_ids)}')

cls_model.eval()
seg_model.eval()

rows = []
for i, img_id in enumerate(test_ids):
    img = np.array(Image.open(os.path.join(TEST_IMG, f'{img_id}.jpg')).convert('RGB'))
    h, w = img.shape[:2]

    cls_result = cls_predict_tta(cls_model, img, best_thresholds)
    seg_mask = seg_predict_tta(seg_model, img, h, w)
    rle = mask_to_rle(seg_mask)

    rows.append({'image_id': img_id, 'classification': cls_result, 'segmentation_rle': rle})

    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(test_ids)}')

submission = pd.DataFrame(rows)
submission.to_csv('output/submission.csv', index=False)
print(f'\nSubmission saved: {len(submission)} rows')
print(submission.head())

Test images: 713
  100/713
  200/713
  300/713
  400/713
  500/713
  600/713
  700/713

Submission saved: 713 rows
    image_id        classification  \
0  img_00005        bicycle person   
1  img_00009          horse person   
2  img_00010  car motorbike person   
3  img_00014                   cat   
4  img_00015          horse person   

                                    segmentation_rle  
0  68707 9 15 69202 18 15 69700 22 15 70199 24 15...  
1  41596 2 13 42095 5 13 42595 6 13 43094 8 13 43...  
2  66267 7 15 66766 9 15 67263 14 15 67762 16 15 ...  
3  28751 21 8 28785 1 8 29240 58 8 29732 89 8 302...  
4  31321 5 15 31635 12 15 31950 18 15 32266 22 15...  


In [18]:
# ---- Cell 21: Sanity check ----
print('Classification distribution:')
bg_count = (submission['classification'] == 'background').sum()
print(f'  background only: {bg_count}/{len(submission)}')
print(f'  with classes: {len(submission) - bg_count}/{len(submission)}')
print(f'\nSegmentation RLE:')
empty_rle = (submission['segmentation_rle'] == '').sum()
print(f'  empty masks: {empty_rle}/{len(submission)}')
print(f'  with masks: {len(submission) - empty_rle}/{len(submission)}')

Classification distribution:
  background only: 6/713
  with classes: 707/713

Segmentation RLE:
  empty masks: 1/713
  with masks: 712/713


---
## Part 4: Write model.py files for submission

In [19]:
# ---- Cell 23: Write classification/model.py (v6: 256px) ----
cls_model_code = '''"""classification/model.py - ResNet50 + GeM + custom head (v6)"""
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

CLS_SIZE = 256

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)

class MultiLabelClassifier(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        backbone = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.gem = GeM(p=3.0)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.gem(self.features(x)))

CLASS_NAMES = ["aeroplane","bicycle","bird","boat","bottle","bus","car","cat",
               "chair","cow","diningtable","dog","horse","motorbike","person",
               "pottedplant","sheep","sofa","train","tvmonitor"]

def predict(image_path, model_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MultiLabelClassifier(num_classes=20).to(device)
    model.load_state_dict(torch.load(os.path.join(model_dir, "weights", "best_model.pth"), map_location=device))
    model.eval()
    thresholds = np.load(os.path.join(model_dir, "weights", "thresholds.npy"))

    transform = transforms.Compose([
        transforms.Resize((CLS_SIZE+32, CLS_SIZE+32)),
        transforms.CenterCrop(CLS_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

    img = Image.open(image_path).convert("RGB")
    inp = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(inp)).squeeze().cpu().numpy()

    classes = [CLASS_NAMES[i] for i in range(20) if probs[i] >= thresholds[i]]
    return " ".join(classes) if classes else "background"
'''

os.makedirs('output/classification', exist_ok=True)
with open('output/classification/model.py', 'w') as f:
    f.write(cls_model_code)
print('classification/model.py written (v6: 256px)')

classification/model.py written (v6: 256px)


In [20]:
# ---- Cell 24: Write segmentation/model.py (v5: ResNet101) ----
seg_model_code = '''"""segmentation/model.py - ResNet101 + ASPP + UNet decoder (v5)"""
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256):
        super().__init__()
        self.conv1x1 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.conv_r6 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=6, dilation=6), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.conv_r12 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=12, dilation=12), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.conv_r18 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=18, dilation=18), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.project = nn.Sequential(nn.Conv2d(out_ch * 5, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout(0.5))
    def forward(self, x):
        h, w = x.shape[2:]
        p = F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=False)
        return self.project(torch.cat([self.conv1x1(x), self.conv_r6(x), self.conv_r12(x), self.conv_r18(x), p], dim=1))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class SegResNet101UNet(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()
        backbone = models.resnet101(weights=None)
        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.aspp = ASPP(2048, 256)
        self.dec4 = DecoderBlock(256, 1024, 256)
        self.dec3 = DecoderBlock(256, 512, 128)
        self.dec2 = DecoderBlock(128, 256, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.final_up = nn.ConvTranspose2d(32, 32, kernel_size=4, stride=2, padding=1)
        self.final_conv = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )
        self.aux_head = nn.Sequential(
            nn.Conv2d(1024, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Conv2d(256, num_classes, 1)
        )
    def forward(self, x):
        e0 = self.layer0(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        x = self.aspp(e4)
        x = self.dec4(x, e3)
        x = self.dec3(x, e2)
        x = self.dec2(x, e1)
        x = self.dec1(x, e0)
        x = F.relu(self.final_up(x))
        return self.final_conv(x)

def predict(image_path, model_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SegResNet101UNet(num_classes=21).to(device)
    model.load_state_dict(torch.load(os.path.join(model_dir, "weights", "best_model.pth"), map_location=device), strict=False)
    model.eval()

    img = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img.size
    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    inp = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(inp).argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    pred_resized = np.array(Image.fromarray(pred).resize((orig_w, orig_h), Image.NEAREST))

    flat = pred_resized.flatten()
    starts, lengths, values = [], [], []
    i = 0
    while i < len(flat):
        if flat[i] != 0:
            val = flat[i]; start = i
            while i < len(flat) and flat[i] == val: i += 1
            starts.append(start); lengths.append(i - start); values.append(val)
        else: i += 1
    if not starts: return ""
    parts = []
    for s, l, v in zip(starts, lengths, values):
        parts.extend([str(s), str(l), str(v)])
    return " ".join(parts)
'''

os.makedirs('output/segmentation', exist_ok=True)
with open('output/segmentation/model.py', 'w') as f:
    f.write(seg_model_code)
print('segmentation/model.py written (ResNet101)')

segmentation/model.py written (ResNet101)


In [21]:
import shutil, numpy as np

# Save thresholds
np.save('/kaggle/working/output/classification/weights/thresholds.npy', np.array(best_thresholds))

# Save submission
submission.to_csv('/kaggle/working/output/submission.csv', index=False)

# Re-zip everything
shutil.make_archive('/kaggle/working/output', 'zip', '/kaggle/working', 'output')
size = os.path.getsize('/kaggle/working/output.zip') / 1e6
print(f'output.zip ready — {size:.0f} MB — download from Data tab')


output.zip ready — 343 MB — download from Data tab
